In [1]:
import pandas as pd
import numpy as np
import matplotlib as mtl
import seaborn as sb

In [2]:
df = pd.read_csv("dataset/Attacks_AppOnly_02-03-2026.csv",sep=';', low_memory=False)

In [3]:
df.shape

(3070159, 52)

In [4]:
df.columns

Index(['id', 'headache', 'migraine', 'attack_time', 'intensity', 'duration',
       'symptoms_both_side_headache', 'symptoms_one_side_headache',
       'symptoms_pulsating_headache', 'symptoms_dull_headache',
       'symptoms_increased_during_physical_activity', 'symptoms_nausea',
       'symptoms_vomiting', 'symptoms_light_sensitivity',
       'symptoms_noise_sensitivity', 'symptoms_smell_sensitivity',
       'symptoms_vertigo', 'symptoms_concentration_disorder',
       'symptoms_tiredness', 'symptoms_aura', 'symptoms_aura_vision',
       'symptoms_aura_emotion', 'symptoms_aura_speech',
       'symptoms_aura_paralysis', 'effects_unfit_for_work',
       'effects_unfit_for_household_recreation', 'effects_doctor',
       'effects_emergency_room', 'effects_hospital', 'comment_menstruation',
       'patient_info_id', 'source_version_id', 'api_version_id', 'create_date',
       'modified_date', 'nMed', 'Triptane_Geb', 'Triptane_Eff',
       'EinfacheSM_Geb', 'EinfacheSM_Eff', 'Uebelkeit_Geb

In [35]:
from ydata_profiling import ProfileReport
report = ProfileReport(df, title="Post-Merge Profile", explorative=True)
report.to_file("profile_report_app_attack.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 52/52 [00:44<00:00,  1.17it/s][A


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
df.duplicated().sum()

np.int64(0)

In [25]:
df.groupby(["patient_info_id"]).size().sort_values(ascending=False)

patient_info_id
45861150    1974
58668734    1937
65018782    1636
37360157    1474
98600514    1350
            ... 
32092068       1
32095145       1
32098888       1
31947019       1
78308651       1
Length: 58811, dtype: int64

In [26]:
relevant_columns=["id","patient_info_id","headache","migraine","attack_time","duration","symptoms_both_side_headache", "symptoms_one_side_headache",
       "symptoms_pulsating_headache", "symptoms_dull_headache","weekday"]
req_df=df[relevant_columns].copy()

In [27]:
req_df.head()

,id,patient_info_id,headache,migraine,attack_time,duration,symptoms_both_side_headache,symptoms_one_side_headache,symptoms_pulsating_headache,symptoms_dull_headache,weekday
0,117,72741515,True,False,2020-06-08,480,False,False,True,False,Montag
1,118,72741515,True,True,2020-06-09,360,False,False,True,False,Dienstag
2,127,44203359,True,True,2020-06-19,300,False,True,True,False,Freitag
3,128,44203359,True,True,2020-06-20,300,False,True,True,False,Samstag
4,129,44203359,True,True,2020-06-21,480,False,True,True,False,Sonntag


In [28]:
#checking the missing value in the df
req_df.isnull().sum()

id                             0
patient_info_id                0
headache                       0
migraine                       0
attack_time                    0
duration                       0
symptoms_both_side_headache    0
symptoms_one_side_headache     0
symptoms_pulsating_headache    0
symptoms_dull_headache         0
weekday                        0
dtype: int64

In [29]:
req_df.isna().any()

id                             False
patient_info_id                False
headache                       False
migraine                       False
attack_time                    False
duration                       False
symptoms_both_side_headache    False
symptoms_one_side_headache     False
symptoms_pulsating_headache    False
symptoms_dull_headache         False
weekday                        False
dtype: bool

In [30]:
req_df.dtypes

id                              int64
patient_info_id                 int64
headache                         bool
migraine                         bool
attack_time                    object
duration                        int64
symptoms_both_side_headache      bool
symptoms_one_side_headache       bool
symptoms_pulsating_headache      bool
symptoms_dull_headache           bool
weekday                        object
dtype: object

In [31]:
#parsed = pd.to_datetime(df["date"], errors="coerce")
#parsed.isna()

req_df["attack_time"]=pd.to_datetime(req_df["attack_time"],format="%Y-%m-%d")

In [32]:
req_df["weekday"].unique()

array(['Montag', 'Dienstag', 'Freitag', 'Samstag', 'Sonntag', 'Mittwoch',
       'Donnerstag'], dtype=object)

In [33]:
order = ["Montag","Dienstag","Mittwoch","Donnerstag","Freitag","Samstag","Sonntag"]
req_df["weekday"] = pd.Categorical(req_df["weekday"], categories=order, ordered=True)
req_df.dtypes

id                                      int64
patient_info_id                         int64
headache                                 bool
migraine                                 bool
attack_time                    datetime64[ns]
duration                                int64
symptoms_both_side_headache              bool
symptoms_one_side_headache               bool
symptoms_pulsating_headache              bool
symptoms_dull_headache                   bool
weekday                              category
dtype: object

In [34]:
#check the numeric columns for irregularity

# Convert to numeric, invalid entries become NaN
req_df["duration_check"] = pd.to_numeric(req_df["duration"], errors='coerce')

# Rows where conversion failed
invalid_rows = req_df[req_df["duration_check"].isna()]
print("Rows with invalid duration values:")
print(invalid_rows)

Rows with invalid duration values:
Empty DataFrame
Columns: [id, patient_info_id, headache, migraine, attack_time, duration, symptoms_both_side_headache, symptoms_one_side_headache, symptoms_pulsating_headache, symptoms_dull_headache, weekday, duration_check]
Index: []


In [35]:
invalid_weekday = req_df[~req_df["weekday"].apply(lambda x: isinstance(x, str))]
print("Rows with non-string weekday:")
print(invalid_weekday)

Rows with non-string weekday:
Empty DataFrame
Columns: [id, patient_info_id, headache, migraine, attack_time, duration, symptoms_both_side_headache, symptoms_one_side_headache, symptoms_pulsating_headache, symptoms_dull_headache, weekday, duration_check]
Index: []


In [36]:
req_df.nunique()

id                             3070159
patient_info_id                  58811
headache                             1
migraine                             2
attack_time                       2072
duration                            25
symptoms_both_side_headache          2
symptoms_one_side_headache           2
symptoms_pulsating_headache          2
symptoms_dull_headache               2
weekday                              7
duration_check                      25
dtype: int64

In [37]:
patient_df = pd.read_csv("dataset/Patients_AppOnly_02-03-2026.csv", on_bad_lines='skip', sep=';', low_memory=False)

In [38]:
rel_columns=['id_patient', 'date_registration',
       'gender', 'birthyear','entries_app', 'first_app_entry', 'last_app_entry',
       'period_use_app','headache_entries_app', 'migraine_entries_app']

In [39]:
req_patient=patient_df[rel_columns].copy()

In [40]:
cols = ["date_registration","first_app_entry","last_app_entry"]

req_patient[cols] = req_patient[cols].apply(pd.to_datetime)

In [41]:
req_patient.head()

,id_patient,date_registration,gender,birthyear,entries_app,first_app_entry,last_app_entry,period_use_app,headache_entries_app,migraine_entries_app
0,10005064,2024-02-01,weiblich,2003,NaN,NaT,NaT,NaN,NaN,NaN
1,10005187,2024-04-27,weiblich,2003,19.0,2024-04-22,2024-05-19,28.0,9.0,0.0
2,10005708,2021-09-17,männlich,1992,53.0,2021-09-08,2021-11-22,76.0,10.0,5.0
3,10005818,2025-03-16,weiblich,1968,24.0,2025-03-08,2025-12-02,270.0,22.0,16.0
4,10009199,2025-09-30,männlich,1984,51.0,2025-09-28,2025-11-18,52.0,24.0,2.0


In [42]:
req_patient.isna().any()

id_patient              False
date_registration       False
gender                  False
birthyear               False
entries_app              True
first_app_entry          True
last_app_entry           True
period_use_app           True
headache_entries_app     True
migraine_entries_app     True
dtype: bool

In [43]:
req_patient.isnull().sum()

id_patient                  0
date_registration           0
gender                      0
birthyear                   0
entries_app             13298
first_app_entry         13298
last_app_entry          13298
period_use_app          13298
headache_entries_app    13298
migraine_entries_app    13298
dtype: int64

In [44]:
df_final = req_df.merge(
    req_patient,
    left_on="patient_info_id",
    right_on="id_patient",
    how="left"
)

In [45]:
df_final.isnull().sum()

id                             0
patient_info_id                0
headache                       0
migraine                       0
attack_time                    0
duration                       0
symptoms_both_side_headache    0
symptoms_one_side_headache     0
symptoms_pulsating_headache    0
symptoms_dull_headache         0
weekday                        0
duration_check                 0
id_patient                     0
date_registration              0
gender                         0
birthyear                      0
entries_app                    0
first_app_entry                0
last_app_entry                 0
period_use_app                 0
headache_entries_app           0
migraine_entries_app           0
dtype: int64

In [46]:
df_final.duplicated()

0          False
1          False
2          False
3          False
4          False
           ...  
3070154    False
3070155    False
3070156    False
3070157    False
3070158    False
Length: 3070159, dtype: bool

In [47]:
df_final.drop(columns=["id_patient","id","duration_check"],inplace=True)
len(df_final) == len(req_df)

True

In [48]:
df_final.columns

Index(['patient_info_id', 'headache', 'migraine', 'attack_time', 'duration',
       'symptoms_both_side_headache', 'symptoms_one_side_headache',
       'symptoms_pulsating_headache', 'symptoms_dull_headache', 'weekday',
       'date_registration', 'gender', 'birthyear', 'entries_app',
       'first_app_entry', 'last_app_entry', 'period_use_app',
       'headache_entries_app', 'migraine_entries_app'],
      dtype='object')

In [49]:
#filter above and equal to 18 years
df_final["age"] = df_final["date_registration"].dt.year - df_final["birthyear"]
df_final = df_final[df_final["age"] >= 18]

In [50]:
len(df_final)  # the patient removed are 99174

2970985

In [52]:
df_final["year"] = df_final["attack_time"].dt.year
df_final["month"] = df_final["attack_time"].dt.month

In [53]:
df_final.head()

,patient_info_id,headache,migraine,attack_time,duration,symptoms_both_side_headache,symptoms_one_side_headache,symptoms_pulsating_headache,symptoms_dull_headache,weekday,...,birthyear,entries_app,first_app_entry,last_app_entry,period_use_app,headache_entries_app,migraine_entries_app,age,year,month
0,72741515,True,False,2020-06-08,480,False,False,True,False,Montag,...,1980,2056.0,2020-06-07,2026-02-15,2080.0,174.0,30.0,40,2020,6
1,72741515,True,True,2020-06-09,360,False,False,True,False,Dienstag,...,1980,2056.0,2020-06-07,2026-02-15,2080.0,174.0,30.0,40,2020,6
2,44203359,True,True,2020-06-19,300,False,True,True,False,Freitag,...,1961,1166.0,2020-06-19,2023-10-17,1216.0,148.0,117.0,59,2020,6
3,44203359,True,True,2020-06-20,300,False,True,True,False,Samstag,...,1961,1166.0,2020-06-19,2023-10-17,1216.0,148.0,117.0,59,2020,6
4,44203359,True,True,2020-06-21,480,False,True,True,False,Sonntag,...,1961,1166.0,2020-06-19,2023-10-17,1216.0,148.0,117.0,59,2020,6


In [54]:
df_final.to_csv("cleaned_App_Attack_data.csv", index=False)